# Fine-tuning QLoRA — Vietnamese Legal Assistant (Buổi 3)

Notebook chạy trên **Google Colab GPU** (miễn phí T4 đủ cho Llama 3.2 3B với QLoRA).

**Trước khi chạy:** `Runtime → Change runtime type → Hardware accelerator → GPU (T4)`.

Notebook này **self-contained** — không cần clone repo. Nó tái tạo đúng pipeline của
`training/` (prepare_data → train_qlora → merge_adapter) trong một chỗ.

Luồng: **Setup → Data → QLoRA Train → Test → Merge → (tùy chọn) Push HF**.

## 0. Kiểm tra GPU

In [ ]:
!nvidia-smi

## 1. Cài đặt dependencies

Cùng bộ thư viện như `training/requirements-train.txt`.

In [ ]:
!pip install -q -U "transformers>=4.44.0" "peft>=0.12.0" "trl>=0.9.0" \
    "datasets>=2.20.0" "accelerate>=0.33.0" "bitsandbytes>=0.43.0"

## 2. Đăng nhập Hugging Face

Llama 3.2 là **gated model** — cần chấp nhận license tại
[huggingface.co/meta-llama/Llama-3.2-3B-Instruct](https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct)
và đăng nhập bằng HF token (Settings → Access Tokens).

In [ ]:
from huggingface_hub import login

login()  # dán HF token khi được hỏi

## 3. Cấu hình

Tương ứng `training/config.py`.

In [ ]:
MODEL_ID = "meta-llama/Llama-3.2-3B-Instruct"
OUTPUT_DIR = "./llama-3.2-3b-legal-vi"
MERGED_DIR = OUTPUT_DIR + "-merged"

# LoRA (Section 3)
LORA_R = 16
LORA_ALPHA = 32          # thường = 2×r
LORA_DROPOUT = 0.05
TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",   # attention
    "gate_proj", "up_proj", "down_proj",       # FFN
]

# Training (Section 5)
NUM_EPOCHS = 3
BATCH_SIZE = 2
GRAD_ACCUM = 4           # effective batch = 2×4 = 8
LEARNING_RATE = 2e-4
MAX_SEQ_LENGTH = 1024

# Push HF (tùy chọn)
PUSH_TO_HUB = False
HUB_REPO_ID = "your-username/llama-3.2-3b-legal-vi"

## 4. Dataset — Legal Q&A (Alpaca format)

Seed nhỏ để minh hoạ pipeline. **Thực tế cần ≥ 100–1000 example chất lượng cao**
(chất lượng > số lượng). Có thể thay bằng dataset HF, ví dụ:
`load_dataset("5CD-AI/Vietnamese-alpaca-gpt4-gg-translated", split="train")`.

In [ ]:
LEGAL_SEED = [
    {"instruction": "Công ty TNHH hai thành viên trở lên có tối đa bao nhiêu thành viên?", "input": "", "output": "Theo Điều 46 Luật Doanh nghiệp 2020, công ty TNHH hai thành viên trở lên có từ 02 đến 50 thành viên. Nếu vượt quá 50, công ty phải chuyển đổi sang loại hình công ty cổ phần."},
    {"instruction": "Công ty cổ phần cần tối thiểu bao nhiêu cổ đông?", "input": "", "output": "Theo Điều 111 Luật Doanh nghiệp 2020, công ty cổ phần phải có tối thiểu 03 cổ đông và không giới hạn số lượng cổ đông tối đa."},
    {"instruction": "Vốn điều lệ là gì?", "input": "", "output": "Vốn điều lệ là tổng giá trị tài sản mà các thành viên/cổ đông đã góp hoặc cam kết góp khi thành lập công ty, được ghi trong Điều lệ công ty (Điều 4 Luật Doanh nghiệp 2020)."},
    {"instruction": "Giải thích trách nhiệm hữu hạn trong công ty TNHH.", "input": "", "output": "Trách nhiệm hữu hạn nghĩa là thành viên công ty TNHH chỉ chịu trách nhiệm về các khoản nợ và nghĩa vụ tài sản khác của công ty trong phạm vi số vốn đã góp vào công ty (Điều 46 Luật Doanh nghiệp 2020)."},
    {"instruction": "Thời hạn góp vốn khi thành lập công ty là bao lâu?", "input": "", "output": "Theo Điều 47 và Điều 113 Luật Doanh nghiệp 2020, thành viên/cổ đông phải góp đủ vốn đã cam kết trong thời hạn 90 ngày kể từ ngày được cấp Giấy chứng nhận đăng ký doanh nghiệp."},
]

# GỢI Ý: trên Colab bạn cũng có thể upload file training/data/legal_seed.jsonl rồi:
#   import json
#   LEGAL_SEED = [json.loads(l) for l in open('legal_seed.jsonl') if l.strip()]

print(f"{len(LEGAL_SEED)} example")

In [ ]:
from datasets import Dataset


def format_prompt(example):
    """Alpaca prompt template (khớp training/prepare_data.py)."""
    if example.get("input"):
        return (
            f"### Instruction:\n{example['instruction']}\n\n"
            f"### Input:\n{example['input']}\n\n"
            f"### Response:\n{example['output']}"
        )
    return (
        f"### Instruction:\n{example['instruction']}\n\n"
        f"### Response:\n{example['output']}"
    )


dataset = Dataset.from_list(LEGAL_SEED)
dataset = dataset.map(lambda x: {"text": format_prompt(x)})
print(dataset[0]["text"])

## 5. Load model 4-bit (QLoRA)

Section 4: quantize NF4 để giảm VRAM → train 3B trên T4 16GB.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False

## 6. Train QLoRA

Section 3 + 5: LoRA adapter + SFTTrainer.

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=TARGET_MODULES,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    bf16=True,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    logging_steps=1,
    save_strategy="epoch",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=True,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
)

trainer.train()
trainer.save_model()
print(f"Adapter đã lưu ở: {OUTPUT_DIR}")

## 7. Test thử adapter

Sinh thử một câu để xem model đã học giọng văn pháp lý chưa.

In [ ]:
prompt = "### Instruction:\nCông ty TNHH một thành viên do ai làm chủ sở hữu?\n\n### Response:\n"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
out = model.generate(**inputs, max_new_tokens=200, do_sample=False)
print(tokenizer.decode(out[0], skip_special_tokens=True))

## 8. Merge adapter → model độc lập

Section 6: `merge_and_unload()` gộp adapter vào base weights → inference không cần PEFT.

> Cần load lại base model ở FP16 (không quantize) để merge chính xác.

In [ ]:
# Giải phóng VRAM của model 4-bit trước khi load base FP16
import gc

del model, trainer
gc.collect()
torch.cuda.empty_cache()

In [ ]:
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
merged = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
merged = merged.merge_and_unload()

merged.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)
print(f"Merged! Model độc lập đã lưu ở: {MERGED_DIR}")

## 9. (Tùy chọn) Push lên Hugging Face Hub

Đặt `PUSH_TO_HUB = True` ở cell cấu hình và sửa `HUB_REPO_ID`.

In [ ]:
if PUSH_TO_HUB:
    merged.push_to_hub(HUB_REPO_ID)
    tokenizer.push_to_hub(HUB_REPO_ID)
    print(f"Đã push: https://huggingface.co/{HUB_REPO_ID}")
else:
    print("Bỏ qua push (PUSH_TO_HUB=False).")

## 10. Deploy — nối lại với app (Buổi 2)

Model đã merge có thể serve qua backend đã dựng ở Buổi 2, rồi app dùng ngay:

```bash
# vLLM (GPU): trỏ thẳng tới HF repo đã push hoặc thư mục merged
vllm serve your-username/llama-3.2-3b-legal-vi

# .env của app:
#   LLM_BACKEND=vllm
#   LLM_MODEL=your-username/llama-3.2-3b-legal-vi
```

Để chạy trên **Ollama**, cần convert model merged sang GGUF (llama.cpp) rồi
`ollama create legal-assistant -f Modelfile` với `FROM ./path-to-gguf`.

→ App (`completion.py`, `pipeline.py`) **không đổi dòng nào**.